In [ ]:
# To make the code work, run the following comands
# python3 -m venv venv
# source venv/bin/activate
# pip install -r requirements.txt

from astropy.table import Table
from astropy.coordinates import SkyCoord
import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from astroquery.gaia import Gaia
import pandas as pd
import os

Gaia.MAIN_GAIA_TABLE = "gaiadr3.gaia_source"

In [ ]:
# Creating the parameters for different stars
# Stars based on star types / Herzsprung - Russel Diagram  

# Make the folder
os.makedirs("./Galah", exist_ok=True)

# https://cloud.datacentral.org.au/teamdata/GALAH/public/GALAH_DR4/catalogs/galah_dr4_allstar_240705.fits
# This is a link to the raw catalogue, I am unable to submit it directly because of its size.
# This box will fail to run without the file. 
# The output from this box has been provided in the Galah folder.

t = Table.read('galah_dr4_allstar_240705.fits')
df = t.to_pandas()

giants = df[(df['teff'] < 5500) & (df['logg'] < 3.0) & (df['flag_sp'] == 0)]
# Constraining parameters to extract giants
print(f"Total giants found: {len(giants)}")
giants_sample = giants.sample(40, random_state=42)
# Extract 40 random giants
giants_sample.to_csv('Galah/giants.csv', index=False)

#repeating for subsequent categories of stars
subgiants = df[(df['teff'].between(5000, 6500)) & (df['logg'].between(3.0, 4.0))]
print(f"Total subgiants found: {len(subgiants)}")
subgiants_sample = subgiants.sample(40, random_state=42)
subgiants_sample.to_csv('Galah/subgiants.csv', index=False)

soba = df[(df['teff'] > 7500) & (df['logg'] > 3.5)]
print(f"Total soba found: {len(soba)}")
soba_sample = soba.sample(90, random_state=40)
# There is a larger sampling of O/B/A stars because a large number of them do not have a full dataset.
soba_sample.to_csv('Galah/soba.csv', index=False)

sfg = df[(df['teff'].between (5200, 7500)) & (df['logg'] > 4.0)]
print(f"Total sfg found: {len(sfg)}")
sfg_sample = sfg.sample(40, random_state=42)
sfg_sample.to_csv('Galah/sfg.csv', index=False)

skm = df[(df['teff'] < 5200) & (df['logg'] > 4.0)]
print(f"Total skm found: {len(skm)}")
skm_sample = skm.sample(40, random_state=42)
skm_sample.to_csv('Galah/skm.csv', index=False)


# Stars based on position in the galaxy

halo = df[(df['fe_h'] < -1.0) & (df['flag_sp'] == 0)]
print(f"Total halo found: {len(halo)}")
halo_sample = halo.sample(40, random_state=42)
halo_sample.to_csv('Galah/halo.csv', index=False)

thick_disk = df[
    (df['fe_h'].between(-1.0, -0.3)) &
    (df['mg_fe'] > 0.2) &
    (df['flag_mg_fe'] == 0) & 
    (df['flag_sp'] == 0) &
    (df['age'] > 8)]
print(f"Total thick disk found: {len(thick_disk)}")
thick_disk_sample = thick_disk.sample(40, random_state=42)
thick_disk_sample.to_csv('Galah/thick_disk.csv', index=False)

young_thin = df[
    (df['fe_h'] > 0.0) &
    (df['mg_fe'] < 0.1) &
    (df['flag_mg_fe'] == 0) &
    (df['flag_sp'] == 0) &
    (df['age'] < 4)]
print(f"Total young thin disk found: {len(young_thin)}")
young_thin_sample = young_thin.sample(40, random_state=42)
young_thin_sample.to_csv('Galah/young_thin_disk.csv', index=False)

old_thin = df[
    (df['fe_h'].between(-0.3, 0.0)) &
    (df['mg_fe'].between(0.1, 0.2)) &
    (df['flag_mg_fe'] == 0) &
    (df['flag_sp'] == 0) &
    (df['age'].between(4, 8))]
print(f"Total old thin disk found :   {len(old_thin)}")
old_thin_sample = old_thin.sample(40, random_state=42)
old_thin_sample.to_csv('Galah/old_thin_disk.csv', index=False)

In [ ]:
# Retrieving data from Gaia for the same set of stars

# Creating a folder
os.makedirs("./GaiaRaw", exist_ok=True)

# Load all stars into a map
galah_stars = {
    "skm": pd.read_csv('Galah/skm.csv'),
    "sfg": pd.read_csv('Galah/sfg.csv'),
    "soba": pd.read_csv('Galah/soba.csv'),
    "giants": pd.read_csv('Galah/giants.csv'),
    "subgiants": pd.read_csv('Galah/subgiants.csv'),
    "halo": pd.read_csv('Galah/halo.csv'),
    "thick_disk": pd.read_csv('Galah/thick_disk.csv'),
    "young_thin_disk": pd.read_csv('Galah/young_thin_disk.csv'),
    "old_thin_disk": pd.read_csv('Galah/old_thin_disk.csv'),
}

print("Retriving Gaia data:")

# Iterate through the dict and query the Gaia Catalogue
for label, data in galah_stars.items():
    # Get Gaia source IDs from Galah
    source_ids = data['gaiadr3_source_id'].dropna().astype(int).tolist()

    ids_str = ','.join(map(str, source_ids))

    # Construct Gaia DR3 query
    query = f"""
    SELECT source_id, ra, dec, distance_gspphot, pmra, pmdec, radial_velocity
    FROM gaiadr3.gaia_source
    WHERE source_id IN ({ids_str})
    AND radial_velocity IS NOT NULL
    """

    job = Gaia.launch_job(query)
    result = job.get_results()

    file_dest = f"GaiaRaw/{label}-gaia.csv"
    result_df = result.to_pandas()
    result_df.to_csv(file_dest, index=False)
    print(f"  - {label}: Saved in {file_dest}")

print("Done")

In [ ]:
# Convert the stars ra, dec and distance into cartesian coordinates.

# Make the folder
os.makedirs("./GalacticCoords", exist_ok=True)

# Load the data into a map
gaia_raw_stars = {
    "skm": pd.read_csv('GaiaRaw/skm-gaia.csv'),
    "sfg": pd.read_csv('GaiaRaw/sfg-gaia.csv'),
    "soba": pd.read_csv('GaiaRaw/soba-gaia.csv'),
    "giants": pd.read_csv('GaiaRaw/giants-gaia.csv'),
    "subgiants": pd.read_csv('GaiaRaw/subgiants-gaia.csv'),
    "halo": pd.read_csv('GaiaRaw/halo-gaia.csv'),
    "thick_disk": pd.read_csv('GaiaRaw/thick_disk-gaia.csv'),
    "young_thin_disk": pd.read_csv('GaiaRaw/young_thin_disk-gaia.csv'),
    "old_thin_disk": pd.read_csv('GaiaRaw/old_thin_disk-gaia.csv'),
}

print(f"Converting to galactic coords:")

# Iterate through the dict and get coords
for label, data in gaia_raw_stars.items():

    # Filter to only rows that have all values
    has_data = data[['ra', 'dec', 'distance_gspphot']].notna().all(axis=1)
    clean = data[has_data].copy()

    print(f"  - {label}")
    print(f"    - Total: {len(data)}")
    print(f"    - With complete data: {len(clean)}")
    print(f"    - Missing distance: {len(data) - len(clean)}")

    # Convert all at once
    coords = SkyCoord(
        ra = clean['ra'].values * u.deg,
        dec = clean['dec'].values * u.deg,
        distance = clean['distance_gspphot'].values * u.pc,
        pm_ra_cosdec = clean['pmra'].values * u.mas/u.yr,
        pm_dec = clean['pmdec'].values * u.mas/u.yr,
        radial_velocity = clean['radial_velocity'].values * u.km/u.s,
        frame = 'icrs',
    )

    clean["l"] = coords.galactic.l.deg
    clean["b"] = coords.galactic.b.deg

    clean['x_pc'] = coords.cartesian.x.to(u.pc).value
    clean['y_pc'] = coords.cartesian.y.to(u.pc).value
    clean['z_pc'] = coords.cartesian.z.to(u.pc).value

    clean["vx"] = coords.galactocentric.v_x.to(u.km/u.s).value
    clean["vy"] = coords.galactocentric.v_y.to(u.km/u.s).value
    clean["vz"] = coords.galactocentric.v_z.to(u.km/u.s).value

    file_dest = f"GalacticCoords/{label}-coords.csv"
    clean.to_csv(file_dest, index=False)
    print(f"    - {label}: Saved in {file_dest}")

print("Done")

In [ ]:
def to_plot_radians(l, b):
    l_plot = np.where(l > 180, l - 360, l)
    return np.radians(l_plot), np.radians(b)

def plot2d(stars_coords, colours):
    fig = plt.figure(figsize=(15, 8))
    ax = fig.add_subplot(111, projection="mollweide")

    for name, df in stars_coords.items():
        l_rad, b_rad = to_plot_radians(df["l"].values, df["b"].values)
        ax.scatter(
            l_rad, b_rad,
            s=10,
            alpha=0.6,
            linewidths=0,
            color=colours[name],
            label=f"{name} (n={len(df):,})",
            rasterized=True,
        )

    # Galactic plane and centre markers
    l_line = np.linspace(-np.pi, np.pi, 500)
    ax.plot(l_line, np.zeros(500), color="gold", lw=0.8, ls="--", alpha=0.6)
    ax.scatter([0], [0], marker="*", s=140, color="yellow", linewidths=0.5, zorder=5)

    # Ticks
    ax.set_xticks(np.radians([-150, -120, -90, -60, -30, 0, 30, 60, 90, 120, 150]))
    ax.set_xticklabels(
        ["210°", "240°", "270°", "300°", "330°", "0°", "30°", "60°", "90°", "120°", "150°"],
        fontsize=7, color="grey",
    )
    ax.set_yticks(np.radians([-60, -30, 0, 30, 60]))
    ax.set_yticklabels(["-60°", "-30°", "0°", "30°", "60°"], fontsize=7, color="grey")

    ax.grid(True, color="grey", lw=0.3, alpha=0.4)
    ax.legend(loc="lower right", fontsize=9, framealpha=0.5, markerscale=2)
    ax.set_title("Mollweide Map (Stellar Groups)", fontsize=13, pad=14)

    plt.show()

In [ ]:
# A 2D representation of the stars in the night sky based on spectral class

stars_coords = {
    "skm": pd.read_csv('GalacticCoords/skm-coords.csv'),
    "sfg": pd.read_csv('GalacticCoords/sfg-coords.csv'),
    "soba": pd.read_csv('GalacticCoords/soba-coords.csv'),
    "giants": pd.read_csv('GalacticCoords/giants-coords.csv'),
    "subgiants": pd.read_csv('GalacticCoords/subgiants-coords.csv'),
}

colours = {
    "skm": "brown",
    "sfg": "yellow",
    "soba": "lightblue",
    "giants": "red",
    "subgiants": "orange",
}

plot2d(stars_coords, colours)

In [ ]:
# A 2D representation of the stars in the night sky based on position

stars_coords = {
    "halo": pd.read_csv('GalacticCoords/halo-coords.csv'),
    "thick_disk": pd.read_csv('GalacticCoords/thick_disk-coords.csv'),
    "young_thin_disk": pd.read_csv('GalacticCoords/young_thin_disk-coords.csv'),
    "old_thin_disk": pd.read_csv('GalacticCoords/old_thin_disk-coords.csv'),
}

colours = {
    "halo": "blue",
    "thick_disk": "darkblue",
    "young_thin_disk": "purple",
    "old_thin_disk": "pink",
}

plot2d(stars_coords, colours)

In [ ]:
# Positions of spectrally divided stars in the sky relative to the sun


def set_visibility(indices):
    vis = [False] * len(fig.data)
    vis[sun_idx] = True

    for idx in indices:
        vis[idx] = True

    return vis

stars_coords = {
    "skm": pd.read_csv('GalacticCoords/skm-coords.csv'),
    "sfg": pd.read_csv('GalacticCoords/sfg-coords.csv'),
    "soba": pd.read_csv('GalacticCoords/soba-coords.csv'),
    "giants": pd.read_csv('GalacticCoords/giants-coords.csv'),
    "subgiants": pd.read_csv('GalacticCoords/subgiants-coords.csv'),
}

colours = {
    "skm": "brown",
    "sfg": "yellow",
    "soba": "lightblue",
    "giants": "red",
    "subgiants": "orange",
}

point_indices = []
cone_indices = []

fig = go.Figure()
for name, data in stars_coords.items():
    point_indices.append(len(fig.data))
    fig.add_trace(go.Scatter3d(
        x = data["x_pc"],
        y = data["y_pc"],
        z = data["z_pc"],
        mode = "markers",
        name = f"{name} (n={len(data):,})",
        marker = dict(
            size = 4,
            color = colours[name],
            opacity = 1,
        ),
        hovertemplate=(
            f'<b>{name}</b><br>'
            'x: %{x:.1f} pc<br>'
            'y: %{y:.1f} pc<br>'
            'z: %{z:.1f} pc'
        ),
    ))

    cone_indices.append(len(fig.data))
    fig.add_trace(go.Cone(
        x = data["x_pc"],
        y = data["y_pc"],
        z = data["z_pc"],
        u = data["vx"],
        v = data["vy"],
        w = data["vz"],
        name=f"{name} velocity",
        legendgroup=name,
        colorscale=[[0, colours[name]], [1, colours[name]]],
        showscale=False,
        sizemode="scaled",
        sizeref=0.5,
        anchor="tail",
        hovertemplate=(
            f"<b>{name}</b><br>"
            "vx: %{u:.1f} km/s<br>"
            "vy: %{v:.1f} km/s<br>"
            "vz: %{w:.1f} km/s<br>"
        ),
    ))

# Sun at origin
sun_idx = len(fig.data)
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode="markers",
    name="Sun",
    marker=dict(size=6, color="yellow", symbol="diamond")
))

fig.update_layout(
    title='3D Star Positions',
    scene=dict(
        xaxis=dict(title="X (pc)", backgroundcolor="black", gridcolor="grey"),
        yaxis=dict(title="Y (pc)", backgroundcolor="black", gridcolor="grey"),
        zaxis=dict(title="Z (pc)", backgroundcolor="black", gridcolor="grey"),

        bgcolor="black",
    ),
    paper_bgcolor="black",
    font_color="white",
    legend=dict(
        itemsizing="constant",
        font=dict(size=11),
    ),
    margin=dict(l=0, r=0, t=40, b=0),

    updatemenus=[dict(
        type="buttons",
        direction="right",
        x=0.5,
        y=1.1,
        xanchor="center",
        yanchor="top",
        buttons=[
            dict(
                label="Show all",
                method="update",
                args=[{"visible": set_visibility(point_indices + cone_indices)},
                      {"title": ""}],
            ),
            dict(
                label="Points",
                method="update",
                args=[{"visible": set_visibility(point_indices)},
                      {"title": ""}],
            ),
            dict(
                label="Velocity",
                method="update",
                args=[{"visible": set_visibility(cone_indices)},
                      {"title": ""}],
            ),
        ],
        bgcolor="grey",
        pad={"r": 10, "t": 10},
    )],
)

fig.write_html("Spectral_plot.html")
fig.show()

In [ ]:
# Positions of location divided stars relative to our sun

def set_visibility(indices):
    vis = [False] * len(fig.data)
    vis[sun_idx] = True

    for idx in indices:
        vis[idx] = True

    return vis

stars_coords = {
    "halo": pd.read_csv('GalacticCoords/halo-coords.csv'),
    "young_thin_disk": pd.read_csv('GalacticCoords/young_thin_disk-coords.csv'),
    "old_thin_disk": pd.read_csv('GalacticCoords/old_thin_disk-coords.csv'),
    "thick_disk": pd.read_csv('GalacticCoords/thick_disk-coords.csv'),
}

colours = {
    "halo": "blue",
    "thick_disk": "darkblue",
    "young_thin_disk": "purple",
    "old_thin_disk": "pink",
}

point_indices = []
cone_indices = []

fig = go.Figure()
for name, data in stars_coords.items():
    point_indices.append(len(fig.data))
    fig.add_trace(go.Scatter3d(
        x = data["x_pc"],
        y = data["y_pc"],
        z = data["z_pc"],
        mode = "markers",
        name = f"{name} (n={len(data):,})",
        marker = dict(
            size = 4,
            color = colours[name],
            opacity = 1,
        ),
        hovertemplate=(
            f'<b>{name}</b><br>'
            'x: %{x:.1f} pc<br>'
            'y: %{y:.1f} pc<br>'
            'z: %{z:.1f} pc'
        ),
    ))

    cone_indices.append(len(fig.data))
    fig.add_trace(go.Cone(
        x = data["x_pc"],
        y = data["y_pc"],
        z = data["z_pc"],
        u = data["vx"],
        v = data["vy"],
        w = data["vz"],
        name=f"{name} velocity",
        legendgroup=name,
        colorscale=[[0, colours[name]], [1, colours[name]]],
        showscale=False,
        sizemode="scaled",
        sizeref=0.5,
        anchor="tail",
        hovertemplate=(
            f"<b>{name}</b><br>"
            "vx: %{u:.1f} km/s<br>"
            "vy: %{v:.1f} km/s<br>"
            "vz: %{w:.1f} km/s<br>"
        ),
    ))

# Sun at Origin
sun_idx = len(fig.data)
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode="markers",
    name="Sun",
    marker=dict(size=6, color="yellow", symbol="diamond")
))

fig.update_layout(
    title='3D Star Positions',
    scene=dict(
        xaxis=dict(title="X (pc)", backgroundcolor="black", gridcolor="grey"),
        yaxis=dict(title="Y (pc)", backgroundcolor="black", gridcolor="grey"),
        zaxis=dict(title="Z (pc)", backgroundcolor="black", gridcolor="grey"),

        bgcolor="black",
    ),
    paper_bgcolor="black",
    font_color="white",
    legend=dict(
        itemsizing="constant",
        font=dict(size=11),
    ),
    margin=dict(l=0, r=0, t=40, b=0),

    updatemenus=[dict(
        type="buttons",
        direction="right",
        x=0.5,
        y=1.1,
        xanchor="center",
        yanchor="top",
        buttons=[
            dict(
                label="Show all",
                method="update",
                args=[{"visible": set_visibility(point_indices + cone_indices)},
                      {"title": ""}],
            ),
            dict(
                label="Points",
                method="update",
                args=[{"visible": set_visibility(point_indices)},
                      {"title": ""}],
            ),
            dict(
                label="Velocity",
                method="update",
                args=[{"visible": set_visibility(cone_indices)},
                      {"title": ""}],
            ),
        ],
        bgcolor="grey",
        pad={"r": 10, "t": 10},
    )],
)

fig.write_html("Position_plot.html")
fig.show()

In [ ]:
# Animation of velocity of spectrally divided stars

stars_coords = {
    "skm": pd.read_csv('GalacticCoords/skm-coords.csv'),
    "sfg": pd.read_csv('GalacticCoords/sfg-coords.csv'),
    "soba": pd.read_csv('GalacticCoords/soba-coords.csv'),
    "giants": pd.read_csv('GalacticCoords/giants-coords.csv'),
    "subgiants": pd.read_csv('GalacticCoords/subgiants-coords.csv'),
}

colours = {
    "skm": "brown",
    "sfg": "yellow",
    "soba": "lightblue",
    "giants": "red",
    "subgiants": "orange",
}

time_steps = np.linspace(0, 1e7, 50)
km_per_pc = 3.086e13
s_per_yr = 3.156e7

frames = []
for t in time_steps:
    frame_data = []
    for name, data in stars_coords.items():
        delta_pc = (data[['vx', 'vy', 'vz']].values * s_per_yr * t) / km_per_pc
        x_new = data['x_pc'].values + delta_pc[:, 0]
        y_new = data['y_pc'].values + delta_pc[:, 1]
        z_new = data['z_pc'].values + delta_pc[:, 2]
        frame_data.append(go.Scatter3d(
            x=x_new, y=y_new, z=z_new,
            mode='markers',
            marker=dict(size=4, color=colours[name], opacity=1),
        ))
    frames.append(go.Frame(data=frame_data, name=str(int(t))))

fig = go.Figure()
for name, data in stars_coords.items():
    fig.add_trace(go.Scatter3d(
        x=data['x_pc'], y=data['y_pc'], z=data['z_pc'],
        mode='markers',
        name=f'{name} (n={len(data):,})',
        marker=dict(size=4, color=colours[name], opacity=1),
    ))

fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers', name='Sun',
    marker=dict(size=6, color='yellow', symbol='diamond')
))

fig.frames = frames
fig.update_layout(
    title='3D Star Positions Over Time — Spectral Types',
    scene=dict(
        xaxis=dict(title='X (pc)', backgroundcolor='black', gridcolor='grey'),
        yaxis=dict(title='Y (pc)', backgroundcolor='black', gridcolor='grey'),
        zaxis=dict(title='Z (pc)', backgroundcolor='black', gridcolor='grey'),
        bgcolor='black',
    ),
    paper_bgcolor='black',
    font_color='white',
    updatemenus=[dict(
        type='buttons', showactive=False,
        y=1.1, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=100, redraw=True), fromcurrent=True, mode='immediate')]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')]),
        ],
        bgcolor='grey',
    )],
    sliders=[dict(
        steps=[
            dict(method='animate',
                 args=[[str(int(t))], dict(mode='immediate', frame=dict(duration=100, redraw=True))],
                 label=f'{int(t/1e6)}Myr')
            for t in time_steps
        ],
        x=0.05, y=0, len=0.9,
        currentvalue=dict(prefix='Time: ', visible=True, xanchor='center', font=dict(color='white')),
        font=dict(color='white'),
    )]
)
fig.write_html('Plot_Animated_Spectral.html')
fig.show()

In [ ]:
# Animation of velocity of location divided stars

stars_coords = {
    "halo": pd.read_csv('GalacticCoords/halo-coords.csv'),
    "thick_disk": pd.read_csv('GalacticCoords/thick_disk-coords.csv'),
    "young_thin_disk": pd.read_csv('GalacticCoords/young_thin_disk-coords.csv'),
    "old_thin_disk": pd.read_csv('GalacticCoords/old_thin_disk-coords.csv'),
}
colours = {
    "halo": "blue",
    "thick_disk": "darkblue",
    "young_thin_disk": "purple",
    "old_thin_disk": "pink",
}

time_steps = np.linspace(0, 1e7, 50)
km_per_pc = 3.086e13
s_per_yr = 3.156e7

frames = []
for t in time_steps:
    frame_data = []
    for name, data in stars_coords.items():
        delta_pc = (data[['vx', 'vy', 'vz']].values * s_per_yr * t) / km_per_pc
        x_new = data['x_pc'].values + delta_pc[:, 0]
        y_new = data['y_pc'].values + delta_pc[:, 1]
        z_new = data['z_pc'].values + delta_pc[:, 2]
        frame_data.append(go.Scatter3d(
            x=x_new, y=y_new, z=z_new,
            mode='markers',
            marker=dict(size=4, color=colours[name], opacity=1),
        ))
    frames.append(go.Frame(data=frame_data, name=str(int(t))))

fig = go.Figure()
for name, data in stars_coords.items():
    fig.add_trace(go.Scatter3d(
        x=data['x_pc'], y=data['y_pc'], z=data['z_pc'],
        mode='markers',
        name=f'{name} (n={len(data):,})',
        marker=dict(size=4, color=colours[name], opacity=1),
    ))

fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0],
    mode='markers', name='Sun',
    marker=dict(size=6, color='yellow', symbol='diamond')
))

fig.frames = frames
fig.update_layout(
    title='3D Star Positions Over Time — Spectral Types',
    scene=dict(
        xaxis=dict(title='X (pc)', backgroundcolor='black', gridcolor='grey'),
        yaxis=dict(title='Y (pc)', backgroundcolor='black', gridcolor='grey'),
        zaxis=dict(title='Z (pc)', backgroundcolor='black', gridcolor='grey'),
        bgcolor='black',
    ),
    paper_bgcolor='black',
    font_color='white',
    updatemenus=[dict(
        type='buttons', showactive=False,
        y=1.1, x=0.5, xanchor='center',
        buttons=[
            dict(label='▶ Play', method='animate',
                 args=[None, dict(frame=dict(duration=100, redraw=True), fromcurrent=True, mode='immediate')]),
            dict(label='⏸ Pause', method='animate',
                 args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')]),
        ],
        bgcolor='grey',
    )],
    sliders=[dict(
        steps=[
            dict(method='animate',
                 args=[[str(int(t))], dict(mode='immediate', frame=dict(duration=100, redraw=True))],
                 label=f'{int(t/1e6)}Myr')
            for t in time_steps
        ],
        x=0.05, y=0, len=0.9,
        currentvalue=dict(prefix='Time: ', visible=True, xanchor='center', font=dict(color='white')),
        font=dict(color='white'),
    )]
)
fig.write_html('Plot_Animated_Position.html')
fig.show()